In [1]:
import pandas as pd
import numpy as np
import requests

In [20]:
# Настройка отображения чисел с плавающей точкой
pd.options.display.float_format = '{:.2f}'.format

df = pd.read_json('../data/auto.json', orient='records')

## Обогащение сэмплом из этого же DataFrame

In [27]:
np.random.seed(21)

unique_cars = df[['CarNumber', 'Make', 'Model']].drop_duplicates()
print(f"Уникальных комбинаций автомобилей: {len(unique_cars)}")

sample_cars = unique_cars.sample(n=200, replace=True, random_state=21)

sample_rows = []
for _, car_row in sample_cars.iterrows():
    # Находим все записи для этой комбинации
    matching_rows = df[(df['CarNumber'] == car_row['CarNumber']) & 
                       (df['Make'] == car_row['Make']) & 
                       (df['Model'] == car_row['Model'])]
    
    if len(matching_rows) > 0:
        random_row = matching_rows.sample(n=1, random_state=np.random.randint(0, 10000))
        new_row = {
            'CarNumber': car_row['CarNumber'],
            'Refund': random_row['Refund'].values[0],
            'Fines': random_row['Fines'].values[0],
            'Make': car_row['Make'],
            'Model': car_row['Model']
        }
        sample_rows.append(new_row)

sample_df = pd.DataFrame(sample_rows)
print(f"\nСоздан сэмпл из {len(sample_df)} наблюдений")

concat_rows = pd.concat([df, sample_df], ignore_index=True)
print(f"Размер concat_rows: {concat_rows.shape}")

Уникальных комбинаций автомобилей: 532

Создан сэмпл из 197 наблюдений
Размер concat_rows: (922, 5)


## Обогащение новой колонкой с годами

In [5]:
# Генерируем Series с годами
np.random.seed(21)
years = np.random.randint(1980, 2020, size=len(concat_rows))
year_series = pd.Series(years, name='Year')

# Конкатенируем с DataFrame
fines = pd.concat([concat_rows, year_series], axis=1)
print("DataFrame после добавления года:")
print(fines.head())
print(f"\nРаспределение годов:")
print(fines['Year'].value_counts().sort_index().head(10))

DataFrame после добавления года:
      CarNumber  Refund   Fines    Make  Model  Year
0  Y163O8161RUS       2 3200.00    Ford  Focus  1989
1   E432XX77RUS       1 6500.00  Toyota  Camry  1995
2   7184TT36RUS       1 2100.00    Ford  Focus  1984
3  X582HE161RUS       2 2000.00    Ford  Focus  2015
4  92918M178RUS       1 5700.00    Ford  Focus  2014

Распределение годов:
Year
1980    21
1981    24
1982    24
1983    19
1984    17
1985    28
1986    17
1987    23
1988    19
1989    27
Name: count, dtype: int64


## Обогащение новой колонкой с годами

In [28]:
# Загружаем фамилии из JSON файла
surnames_data = pd.read_json('../../datasets/surname.json')

if 'name' in surnames_data.columns:
    surnames = surnames_data['name'].tolist()
elif 'surname' in surnames_data.columns:
    surnames = surnames_data['surname'].tolist()
else:
    # Если структура другая, берем первый столбец
    surnames = surnames_data.iloc[:, 0].tolist()

# Очищаем фамилии от спецсимволов
surnames = [str(s).strip().replace(',', '').replace('(', '').replace(')', '').replace('[', '').replace(']', '') 
            for s in surnames if pd.notna(s)]
surnames = list(dict.fromkeys(surnames))  # Удаляем дубликаты

print(f"Загружено {len(surnames)} уникальных фамилий")

Загружено 101 уникальных фамилий


In [7]:
# Создаем Series с фамилиями для уникальных номеров автомобилей
np.random.seed(21)
unique_car_numbers = fines['CarNumber'].unique()
num_unique = len(unique_car_numbers)

# Выбираем случайные фамилии
selected_surnames = np.random.choice(surnames, size=num_unique, replace=(num_unique > len(surnames)))

# Создаем DataFrame owners
owners = pd.DataFrame({
    'CarNumber': unique_car_numbers,
    'SURNAME': selected_surnames
})

print("DataFrame owners:")
print(owners.head())
print(f"\nРазмер owners: {owners.shape}")

DataFrame owners:
      CarNumber    SURNAME
0  Y163O8161RUS   Martinez
1   E432XX77RUS  Rodriguez
2   7184TT36RUS      Jones
3  X582HE161RUS      Smith
4  92918M178RUS      Smith

Размер owners: (531, 2)


## Добавление новых наблюдений

In [8]:
# Добавляем 5 новых наблюдений в fines
new_fines = pd.DataFrame([
    {'CarNumber': 'NEW001RUS', 'Refund': 0.0, 'Fines': 5000.0, 'Make': 'Tesla', 'Model': 'Model S', 'Year': 2020},
    {'CarNumber': 'NEW002RUS', 'Refund': 1.0, 'Fines': 3000.0, 'Make': 'BMW', 'Model': 'X5', 'Year': 2019},
    {'CarNumber': 'NEW003RUS', 'Refund': 2.0, 'Fines': 7000.0, 'Make': 'Audi', 'Model': 'A6', 'Year': 2018},
    {'CarNumber': 'NEW004RUS', 'Refund': 0.0, 'Fines': 2000.0, 'Make': 'Mercedes', 'Model': 'E-Class', 'Year': 2019},
    {'CarNumber': 'NEW005RUS', 'Refund': 1.0, 'Fines': 4500.0, 'Make': 'Lexus', 'Model': 'RX', 'Year': 2020}
])

fines = pd.concat([fines, new_fines], ignore_index=True)
print(f"Fines после добавления 5 новых записей: {fines.shape}")

Fines после добавления 5 новых записей: (927, 6)


In [9]:
# Удаляем последние 20 наблюдений из owners и добавляем 3 новых
owners = owners.iloc[:-20] if len(owners) > 20 else owners

new_owners = pd.DataFrame([
    {'CarNumber': 'OWNER001RUS', 'SURNAME': 'Anderson'},
    {'CarNumber': 'OWNER002RUS', 'SURNAME': 'Thomas'},
    {'CarNumber': 'OWNER003RUS', 'SURNAME': 'Jackson'}
])

owners = pd.concat([owners, new_owners], ignore_index=True)
print(f"Owners после удаления и добавления: {owners.shape}")

Owners после удаления и добавления: (514, 2)


## Соединение DataFrame

In [10]:
# 1. Только номера, которые есть в обоих DataFrame (inner join)
inner_join = pd.merge(fines, owners, on='CarNumber', how='inner')
print("1. Inner join (только в обоих):")
print(f"Размер: {inner_join.shape}")
print(inner_join.head())

1. Inner join (только в обоих):
Размер: (894, 7)
      CarNumber  Refund   Fines    Make  Model  Year    SURNAME
0  Y163O8161RUS    2.00 3200.00    Ford  Focus  1989   Martinez
1   E432XX77RUS    1.00 6500.00  Toyota  Camry  1995  Rodriguez
2   7184TT36RUS    1.00 2100.00    Ford  Focus  1984      Jones
3  X582HE161RUS    2.00 2000.00    Ford  Focus  2015      Smith
4  92918M178RUS    1.00 5700.00    Ford  Focus  2014      Smith


In [11]:
# 2. Все номера из обоих DataFrame (outer join)
outer_join = pd.merge(fines, owners, on='CarNumber', how='outer')
print("\n2. Outer join (все номера):")
print(f"Размер: {outer_join.shape}")
print(outer_join.head())


2. Outer join (все номера):
Размер: (930, 7)
      CarNumber  Refund   Fines  Make  Model    Year   SURNAME
0  704687163RUS    2.00 1400.00  Ford  Focus 2004.00    Miller
1  704787163RUS    2.00 2800.00  Ford  Focus 1992.00    Garcia
2  704987163RUS    2.00 8594.59  Ford  Focus 1985.00  Williams
3  705287163RUS    2.00 2000.00  Ford  Focus 1980.00     Brown
4  705387163RUS    2.00  700.00  Ford  Focus 1987.00    Miller


In [12]:
# 3. Только номера из fines (left join)
left_join = pd.merge(fines, owners, on='CarNumber', how='left')
print("\n3. Left join (только из fines):")
print(f"Размер: {left_join.shape}")
print(left_join.head())


3. Left join (только из fines):
Размер: (927, 7)
      CarNumber  Refund   Fines    Make  Model  Year    SURNAME
0  Y163O8161RUS    2.00 3200.00    Ford  Focus  1989   Martinez
1   E432XX77RUS    1.00 6500.00  Toyota  Camry  1995  Rodriguez
2   7184TT36RUS    1.00 2100.00    Ford  Focus  1984      Jones
3  X582HE161RUS    2.00 2000.00    Ford  Focus  2015      Smith
4  92918M178RUS    1.00 5700.00    Ford  Focus  2014      Smith


In [13]:
# 4. Только номера из owners (right join)
right_join = pd.merge(fines, owners, on='CarNumber', how='right')
print("\n4. Right join (только из owners):")
print(f"Размер: {right_join.shape}")
print(right_join.head())


4. Right join (только из owners):
Размер: (897, 7)
      CarNumber  Refund   Fines    Make  Model    Year    SURNAME
0  Y163O8161RUS    2.00 3200.00    Ford  Focus 1989.00   Martinez
1  Y163O8161RUS    2.00 1600.00    Ford  Focus 1980.00   Martinez
2  Y163O8161RUS    2.00 3200.00    Ford  Focus 1980.00   Martinez
3  Y163O8161RUS    2.00 1600.00    Ford  Focus 1982.00   Martinez
4   E432XX77RUS    1.00 6500.00  Toyota  Camry 1995.00  Rodriguez


## Создание сводной таблицы

In [14]:
# Создаем сводную таблицу: годы как столбцы, марки как строки, значения - суммы штрафов
pivot_table = pd.pivot_table(
    fines,
    values='Fines',
    index='Make',
    columns='Year',
    aggfunc='sum',
    fill_value=0
)

print("Сводная таблица (суммы штрафов по маркам и годам):")
print(pivot_table)

Сводная таблица (суммы штрафов по маркам и годам):
Year           1980      1981      1982     1983      1984      1985     1986  \
Make                                                                            
Audi           0.00      0.00      0.00     0.00      0.00      0.00     0.00   
BMW            0.00      0.00      0.00     0.00      0.00      0.00     0.00   
Ford       65589.17 390183.76 145383.76 67100.00 112389.17 140683.76 93994.59   
Lexus          0.00      0.00      0.00     0.00      0.00      0.00     0.00   
Mercedes       0.00      0.00      0.00     0.00      0.00      0.00     0.00   
Skoda       1900.00      0.00   6900.00 11594.59      0.00  10294.59   600.00   
Tesla          0.00      0.00      0.00     0.00      0.00      0.00     0.00   
Toyota     12000.00   8594.59   6400.00  7200.00      0.00      0.00     0.00   
Volkswagen 30900.00   1600.00      0.00 11794.59  10300.00  34800.00 22400.00   
Volvo          0.00      0.00   6800.00     0.00      0.00

In [15]:
# Сохраняем сводную таблицу в файл для просмотра
pivot_table.to_csv('../data/pivot_table.csv')
print("\nСводная таблица сохранена в data/pivot_table.csv")


Сводная таблица сохранена в data/pivot_table.csv


In [16]:
# Отображаем только годы с 1980 по 1984 как в примере
print("\nСводная таблица (годы 1980-1984):")
if 1980 in pivot_table.columns and 1984 in pivot_table.columns:
    pivot_sample = pivot_table.loc[:, 1980:1984]
    print(pivot_sample)


Сводная таблица (годы 1980-1984):
Year           1980      1981      1982     1983      1984
Make                                                      
Audi           0.00      0.00      0.00     0.00      0.00
BMW            0.00      0.00      0.00     0.00      0.00
Ford       65589.17 390183.76 145383.76 67100.00 112389.17
Lexus          0.00      0.00      0.00     0.00      0.00
Mercedes       0.00      0.00      0.00     0.00      0.00
Skoda       1900.00      0.00   6900.00 11594.59      0.00
Tesla          0.00      0.00      0.00     0.00      0.00
Toyota     12000.00   8594.59   6400.00  7200.00      0.00
Volkswagen 30900.00   1600.00      0.00 11794.59  10300.00
Volvo          0.00      0.00   6800.00     0.00      0.00


## Сохранение DataFrame

In [19]:
# Сохраняем fines и owners в CSV без индекса
fines.to_csv('../data/fines.csv', index=False)
owners.to_csv('../data/owners.csv', index=False)

print("DataFrame сохранены:")
print(f"- fines.csv: {fines.shape}")
print(f"- owners.csv: {owners.shape}")

DataFrame сохранены:
- fines.csv: (927, 6)
- owners.csv: (514, 2)
